# K-Nearest Neighbour On MNIST Data

The objective of this dataset is to accurately classify handwritten digits from 0 to 9. Instead of using the full MNIST dataset, which contains 60,000 training images and 10,000 testing images, we will work with a smaller subset provided by the scikit-learn library. This subset includes 1,797 digit images, which we will divide into training, validation, and testing sets.
Each image is originally an 8 x 8 grayscale image, but scikit-learn converts it into a flattened list.

In [ ]:
# import the necessary packages
from __future__ import print_function
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn import datasets
from skimage import exposure
import numpy as np
import cv2
import imutils
import sklearn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load Mnist data digits 
mnist = datasets.load_digits()

In [ ]:
# take the MNIST data and construct the training and testing split, using 75% of the
# data for training and 25% for testing
(trainData, testData, trainLabels, testLabels) = train_test_split(np.array(mnist.data),
	mnist.target, test_size=0.25, random_state=42)

In [ ]:
# now, let's take 10% of the training data and use that for validation
(trainData, valData, trainLabels, valLabels) = train_test_split(trainData, trainLabels,
	test_size=0.1, random_state=84)

In [ ]:
# show the sizes of each data split
print("training data points: {}".format(len(trainLabels)))
print("validation data points: {}".format(len(valLabels)))
print("testing data points: {}".format(len(testLabels)))

Now that we have our data splits taken care of, let's train our classifier and find the optimal value of k.


In [ ]:
# initialize the values of k for our k-Nearest Neighbor classifier along with the
# list of accuracies for each value of k
kVals = range(1, 30, 2)
accuracies = []

In [ ]:
# loop over various values of `k` for the k-Nearest Neighbor classifier
for k in range(1, 30, 2):
    # train the k-Nearest Neighbor classifier with the current value of `k`
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(trainData, trainLabels)
    # After our model is trained, we need to evaluate it using our validation data    
    # evaluate the model and update the accuracies list
    score = model.score(valData, valLabels)
    print("k=%d, accuracy=%.2f%%" % (k, score * 100))
    accuracies.append(score)

In [ ]:
# find the value of k that has the largest accuracy
i = int(np.argmax(accuracies))
kVals_list = list(kVals)  # Convert range to list for indexing
print("k=%d achieved highest accuracy of %.2f%% on validation data" % (kVals_list[i],
accuracies[i] * 100))

Although the accuracy for \( k = 1 \) through \( k = 15 \) remained consistent, using just one neighbor significantly improves efficiency. Therefore, we will use \( k = 1 \) for training and evaluating our classifier on the final test data.

In [ ]:
# re-train our classifier using the best k value and predict the labels of the
# test data
model = KNeighborsClassifier(n_neighbors=kVals_list[i])
model.fit(trainData, trainLabels)
predictions = model.predict(testData)

In [ ]:
# show a final classification report demonstrating the accuracy of the classifier
# for each of the digits
print("EVALUATION ON TESTING DATA")
print(classification_report(testLabels, predictions))

Achieving 98% accuracy is impressive! Additionally, digits 0, 2, 6, and 7 are classified correctly 100% of the time. The digit 1 has the lowest classification accuracy, at 95%.
Achieving high accuracy on the MNIST dataset doesn't mean handwritten digit recognition is "solved." Despite MNIST being a standard benchmark, its images are heavily pre-processed—cropped, thresholded, and centered—which doesn't reflect real-world conditions. In practice, real-world datasets are often less clean and require feature extraction beyond raw pixel intensities. Nonetheless, this exercise demonstrates how Euclidean distance can yield high accuracy with well-pre-processed data.

Let's conclude this code example by reviewing some individual predictions made by our k-NN classifier.

In [ ]:
# Loop over a few random digits and display predictions
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

random_indices = np.random.randint(0, high=len(testLabels), size=6)

for idx, ax in enumerate(axes):
    i = random_indices[idx]
    
    # Grab the image and classify it
    image = testData[i]
    prediction = model.predict(image.reshape(1, -1))[0]
    actual = testLabels[i]
    
    # Convert the image from a 64-dim array to an 8 x 8 image
    image_reshaped = image.reshape((8, 8)).astype("uint8")
    image_rescaled = exposure.rescale_intensity(image_reshaped, out_range=(0, 255))
    
    # Display the image
    ax.imshow(image_rescaled, cmap='gray')
    
    # Add title with prediction and actual label
    color = 'green' if prediction == actual else 'red'
    ax.set_title(f'Pred: {prediction}, Actual: {actual}', color=color, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()